# KaggleSimEnv — GRPO Training with Unsloth + TRL

Train an LLM agent to solve Kaggle competition simulation tasks using **GRPO** (Group Relative Policy Optimisation).

The environment exposes a REST API. The model generates a complete action plan for a given task,
the plan is executed step-by-step against the env, and the final grade score is used as the reward signal.

**Architecture**
```
LLM ──► action plan (JSON list) ──► POST /reset + POST /step × N + POST /grader ──► reward
  ▲                                                                                      │
  └──────────────────── GRPO update (TRL GRPOTrainer) ◄─────────────────────────────────┘
```

**Requirements:** T4 GPU (free Colab tier is sufficient)

**Environment:** KaggleSimEnv — either run locally (`python -m uvicorn server.app:app`) or use the deployed HF Space URL.

## 1. Install dependencies

In [ ]:
# Install Unsloth (fastest fine-tuning on Colab T4)
!pip install unsloth[colab-new] -q

# Install TRL (GRPO), datasets, matplotlib
!pip install trl>=0.12.0 datasets>=3.0.0 accelerate>=1.0.0 peft>=0.13.0 matplotlib requests -q

print("✅ Dependencies installed")

## 2. Configuration

In [ ]:
import os

# ── ENV URL ──────────────────────────────────────────────────────────────
# For local dev:  ENV_URL = "http://127.0.0.1:7860"
# For HF Space:   ENV_URL = "https://<your-username>-kaggle-sim-env.hf.space"
ENV_URL = os.getenv("ENV_URL", "https://arry7868-kaggle-simulation-environment.hf.space").rstrip("/")

# ── MODEL ────────────────────────────────────────────────────────────────
# Small model that fits on free T4 with Unsloth 4-bit quant
MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"

# ── TRAINING HYPERPARAMS ─────────────────────────────────────────────────
NUM_TRAIN_EPISODES  = 60    # total env episodes for training data
NUM_EVAL_EPISODES   = 10    # episodes for evaluation
GRPO_EPOCHS         = 1
BATCH_SIZE          = 2
GRAD_ACCUMULATION   = 4
LEARNING_RATE       = 5e-5
MAX_NEW_TOKENS      = 512   # max tokens for action plan
NUM_GENERATIONS     = 4     # GRPO group size G

# Tasks to train on (start simple, add harder tasks)
TRAIN_TASKS = ["easy_churn", "medium_fraud"]
EVAL_TASKS  = ["easy_churn", "medium_fraud", "hard_leaky_noisy"]

PLOTS_DIR = "plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

print(f"ENV_URL  : {ENV_URL}")
print(f"Model    : {MODEL_NAME}")
print(f"Episodes : {NUM_TRAIN_EPISODES} train / {NUM_EVAL_EPISODES} eval")

## 3. Verify environment connectivity

In [ ]:
import requests, json

def env_get(path):
    r = requests.get(f"{ENV_URL}{path}", timeout=30)
    r.raise_for_status()
    return r.json()

def env_post(path, body=None):
    r = requests.post(f"{ENV_URL}{path}", json=body or {}, timeout=60)
    r.raise_for_status()
    return r.json()

# Sanity check
health = env_get("/health")
tasks  = env_get("/tasks")

print(f"Health : {health}")
print(f"Tasks  : {[t['task_id'] for t in tasks]}")
assert health.get("status") == "healthy", "Environment not healthy!"
print("\n✅ Environment is reachable and healthy")

## 4. Define prompts and action parsing

In [ ]:
import re
from typing import Any

SYSTEM_PROMPT = """\
You are an expert Kaggle Grand Master. Given a dataset description and task,
generate a complete action plan as a JSON array of actions.

Each action must be a JSON object with this structure:
  {"action_type": "<type>", "parameters": {"category": "<cat>", "<key>": "<value>"}}

Available action types and their categories:
  set_cv:            standard(kfold) | group(group_kfold,stratified_group_kfold) | temporal(time_split)
  feature_engineering: distribution(log_transform,normalize) | interaction(domain_ratios) | encoding(sin_cos_encoding)
  detect_shift:      detection(adversarial_validation) | mitigation(remove_identifiers,domain_invariant_features)
  train_model:       tree(xgboost,lightgbm,catboost) | neural(pretrained_backbone,transformer_encoder)
  handle_imbalance:  weighting(scale_pos_weight) | calibration(optimize_threshold,calibrate_probabilities)
  clean_data:        removal(remove_leaky_features,remove_outliers) | reconstruction(analytical_reconstruction)
  augmentation:      geometric(geometric,rotation_invariant) | color(color_transform,clahe) | domain(camera_simulation)
  ensemble:          averaging(weighted_average,multi_seed_averaging) | stacking(stacking)
  postprocess:       calibration(prediction_shrinkage) | inference(tta) | domain(physics_constraints)
  regularize:        weight(strong_regularization,ema) | transfer(freeze_backbone)
  pseudo_label:      (iterations: 1/2/3)
  inspect_top_solution: {}
  submit:            {}

Rules:
1. ALWAYS start with inspect_top_solution to get hints.
2. Use set_cv before train_model.
3. ALWAYS end with submit.
4. Keep plan to 8-12 actions.
5. Use strategies relevant to the dataset properties described.
6. Respond with ONLY the JSON array, no other text.
"""


def build_task_prompt(task_id: str, task_info: dict) -> str:
    """Format a task as a user prompt for the model."""
    return (
        f"Task: {task_id}\n"
        f"Title: {task_info['title']}\n"
        f"Difficulty: {task_info['difficulty']}\n"
        f"Description: {task_info['description']}\n\n"
        "Generate a complete action plan (JSON array) to score as high as possible:"
    )


def parse_action_plan(text: str) -> list[dict]:
    """Extract JSON array of actions from model output."""
    text = text.strip()
    # Strip markdown code fences
    text = re.sub(r"^```[\w]*\n?", "", text)
    text = re.sub(r"\n?```$", "", text.strip())
    # Find JSON array
    match = re.search(r"\[.*\]", text, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return []


def execute_plan(task_id: str, plan: list[dict]) -> float:
    """
    Execute an action plan against the env.
    Returns the final grade score in [0.0, 1.0].
    """
    try:
        env_post("/reset", {"task_id": task_id})
        for action in plan:
            if not isinstance(action, dict) or "action_type" not in action:
                continue
            params = action.get("parameters", {})
            env_post("/step", {"action": {"action_type": action["action_type"], "parameters": params}})
        grade = env_post("/grader")
        return float(grade.get("final_score", 0.0))
    except Exception as e:
        print(f"  [execute_plan error] {e}")
        return 0.0


# Quick test
task_info = {t["task_id"]: t for t in tasks}
sample_prompt = build_task_prompt("easy_churn", task_info["easy_churn"])
print("Sample prompt:\n")
print(sample_prompt)

## 5. Load model with Unsloth

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=1024,
    dtype=None,            # auto-detect float16 / bfloat16
    load_in_4bit=True,
)

# Add LoRA adapters for GRPO fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 6. Build training dataset

Generate prompts from the available tasks. Each prompt = one task description.
We repeat tasks to fill `NUM_TRAIN_EPISODES` so the model sees diverse orderings.

In [ ]:
from datasets import Dataset
import random

random.seed(42)

def make_chat_prompt(task_id: str) -> list[dict]:
    """Return a messages list suitable for apply_chat_template."""
    return [
        {"role": "system",  "content": SYSTEM_PROMPT},
        {"role": "user",    "content": build_task_prompt(task_id, task_info[task_id])},
    ]

# Build prompt list: cycle through train tasks to reach NUM_TRAIN_EPISODES
prompts_raw = []
task_ids_repeated = (TRAIN_TASKS * (NUM_TRAIN_EPISODES // len(TRAIN_TASKS) + 1))[:NUM_TRAIN_EPISODES]
random.shuffle(task_ids_repeated)

for tid in task_ids_repeated:
    msgs = make_chat_prompt(tid)
    prompt_text = tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=True
    )
    prompts_raw.append({"prompt": prompt_text, "task_id": tid})

train_dataset = Dataset.from_list(prompts_raw)

print(f"✅ Training dataset: {len(train_dataset)} prompts")
print(f"   Task distribution: { {t: task_ids_repeated.count(t) for t in TRAIN_TASKS} }")

## 7. Define reward function

The reward function:
1. Parses the model's JSON output into an action plan
2. Executes the plan step-by-step against the env
3. Returns the final grade score as the reward

In [ ]:
# Track rewards for plotting
_reward_log: list[float] = []
_step_counter = [0]


def kaggle_env_reward(completions: list[str], prompts: list[str], task_id: list[str] = None, **kwargs) -> list[float]:
    """
    GRPO reward function.
    completions : list of model-generated strings (one per sample in the group)
    prompts     : list of input prompts
    task_id     : list of task IDs (passed through dataset columns)
    Returns list of float rewards in [0.0, 1.0].
    """
    rewards = []

    for i, completion in enumerate(completions):
        tid = (task_id[i] if task_id else None) or "easy_churn"
        plan = parse_action_plan(completion)

        if not plan:
            # Malformed output: give 0 reward
            rewards.append(0.0)
            continue

        # Make sure there's a submit at the end
        if not plan or plan[-1].get("action_type") != "submit":
            plan.append({"action_type": "submit", "parameters": {}})

        score = execute_plan(tid, plan)
        rewards.append(score)

        _step_counter[0] += 1
        _reward_log.append(score)

        if _step_counter[0] % 10 == 0:
            avg = sum(_reward_log[-20:]) / min(20, len(_reward_log))
            print(f"  [step {_step_counter[0]:4d}] task={tid}  score={score:.4f}  avg20={avg:.4f}")

    return rewards


# Quick smoke-test of the reward function (uses a hard-coded expert plan)
_test_plan = json.dumps([
    {"action_type": "inspect_top_solution", "parameters": {}},
    {"action_type": "set_cv", "parameters": {"category": "standard", "strategy": "kfold"}},
    {"action_type": "train_model", "parameters": {"category": "tree", "algorithm": "xgboost"}},
    {"action_type": "handle_imbalance", "parameters": {"category": "weighting", "method": "scale_pos_weight"}},
    {"action_type": "submit", "parameters": {}},
])

test_rewards = kaggle_env_reward([_test_plan], ["dummy"], task_id=["easy_churn"])
print(f"\n✅ Smoke test reward: {test_rewards[0]:.4f}  (expected > 0.2)")

## 8. Baseline evaluation (before training)

In [ ]:
import torch

FastLanguageModel.for_inference(model)  # enable fast inference mode


def evaluate_model(label: str = "model", n_episodes: int = NUM_EVAL_EPISODES) -> dict[str, float]:
    """Run n_episodes per eval task, return mean score per task."""
    results: dict[str, list[float]] = {t: [] for t in EVAL_TASKS}

    for tid in EVAL_TASKS:
        msgs = make_chat_prompt(tid)
        prompt_text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

        for ep in range(n_episodes):
            with torch.no_grad():
                out = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=0.7,
                    pad_token_id=tokenizer.eos_token_id,
                )
            completion = tokenizer.decode(
                out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
            )
            plan = parse_action_plan(completion)
            if plan and plan[-1].get("action_type") != "submit":
                plan.append({"action_type": "submit", "parameters": {}})
            score = execute_plan(tid, plan) if plan else 0.0
            results[tid].append(score)
            print(f"  [{label}] {tid} ep{ep+1}: {score:.4f}")

    return {tid: sum(v) / len(v) for tid, v in results.items()}


print("=" * 60)
print("BASELINE evaluation (untrained model)")
print("=" * 60)
baseline_scores = evaluate_model(label="baseline", n_episodes=3)
print("\nBaseline mean scores:")
for tid, score in baseline_scores.items():
    print(f"  {tid:<25} {score:.4f}")
baseline_overall = sum(baseline_scores.values()) / len(baseline_scores)
print(f"  {'MEAN':<25} {baseline_overall:.4f}")

## 9. GRPO Training

In [ ]:
from trl import GRPOConfig, GRPOTrainer

FastLanguageModel.for_training(model)  # switch back to training mode

grpo_config = GRPOConfig(
    output_dir="./grpo-kaggle-agent",
    num_train_epochs=GRPO_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    max_new_tokens=MAX_NEW_TOKENS,
    num_generations=NUM_GENERATIONS,
    temperature=0.9,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    seed=42,
    # GRPO-specific
    kl_coef=0.05,
    reward_baseline=0.3,   # env score for a naive random agent ~0.2
)

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    reward_funcs=[kaggle_env_reward],
    train_dataset=train_dataset,
)

print("✅ GRPOTrainer ready")
print(f"   Dataset size   : {len(train_dataset)}")
print(f"   Epochs         : {GRPO_EPOCHS}")
print(f"   Batch size     : {BATCH_SIZE} (× {GRAD_ACCUMULATION} accum = {BATCH_SIZE * GRAD_ACCUMULATION} effective)")
print(f"   Num generations: {NUM_GENERATIONS}")
print(f"   Learning rate  : {LEARNING_RATE}")

In [ ]:
# ── Run training ─────────────────────────────────────────────────────────
print("Starting GRPO training...")
_reward_log.clear()
_step_counter[0] = 0

train_result = trainer.train()

print("\n✅ Training complete")
print(f"   Train loss : {train_result.training_loss:.4f}")
print(f"   Total steps: {train_result.global_step}")

## 10. Post-training evaluation

In [ ]:
FastLanguageModel.for_inference(model)

print("=" * 60)
print("POST-TRAINING evaluation")
print("=" * 60)
trained_scores = evaluate_model(label="trained", n_episodes=5)

print("\n" + "=" * 60)
print(f"{'Task':<25} {'Baseline':>10} {'Trained':>10} {'Delta':>10}")
print("-" * 60)
for tid in EVAL_TASKS:
    b = baseline_scores.get(tid, 0.0)
    t = trained_scores.get(tid, 0.0)
    delta = t - b
    print(f"{tid:<25} {b:>10.4f} {t:>10.4f} {delta:>+10.4f}")
print("-" * 60)
trained_overall = sum(trained_scores.values()) / len(trained_scores)
delta_overall   = trained_overall - baseline_overall
print(f"{'MEAN':<25} {baseline_overall:>10.4f} {trained_overall:>10.4f} {delta_overall:>+10.4f}")

## 11. Save training plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

plt.rcParams.update({"figure.dpi": 130, "font.size": 11})


# ── Plot 1: Reward curve during training ─────────────────────────────────
def smooth(x, w=10):
    return np.convolve(x, np.ones(w) / w, mode="valid")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(_reward_log, alpha=0.35, color="steelblue", linewidth=0.8, label="Raw episode score")
if len(_reward_log) >= 10:
    ax.plot(range(9, len(_reward_log)), smooth(_reward_log), color="steelblue",
            linewidth=2.0, label="Smoothed (w=10)")
ax.set_xlabel("Training step")
ax.set_ylabel("Episode score (0–1)")
ax.set_title("KaggleSimEnv — GRPO Training Reward Curve")
ax.legend()
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.2f"))
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/reward_curve.png")
plt.show()
print(f"Saved → {PLOTS_DIR}/reward_curve.png")


# ── Plot 2: Training loss (from trainer logs) ────────────────────────────
loss_log = [(e["step"], e["loss"]) for e in trainer.state.log_history if "loss" in e]
if loss_log:
    steps, losses = zip(*loss_log)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(steps, losses, color="tomato", linewidth=2.0)
    ax.set_xlabel("Training step")
    ax.set_ylabel("GRPO Loss")
    ax.set_title("KaggleSimEnv — GRPO Training Loss")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(f"{PLOTS_DIR}/loss_curve.png")
    plt.show()
    print(f"Saved → {PLOTS_DIR}/loss_curve.png")


# ── Plot 3: Baseline vs Trained comparison ───────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
task_labels = list(EVAL_TASKS)
x = np.arange(len(task_labels))
width = 0.35
b_vals = [baseline_scores.get(t, 0) for t in task_labels]
t_vals = [trained_scores.get(t, 0) for t in task_labels]

bars_b = ax.bar(x - width/2, b_vals, width, label="Untrained baseline", color="#aec6cf", edgecolor="black")
bars_t = ax.bar(x + width/2, t_vals, width, label="GRPO trained",       color="#5f9ea0", edgecolor="black")

ax.set_xlabel("Task")
ax.set_ylabel("Mean final score (0–1)")
ax.set_title("KaggleSimEnv — Baseline vs GRPO Trained Agent")
ax.set_xticks(x)
ax.set_xticklabels(task_labels, rotation=15, ha="right")
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis="y", alpha=0.3)

# Annotate bars
for bar in bars_b:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)
for bar in bars_t:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

fig.tight_layout()
fig.savefig(f"{PLOTS_DIR}/baseline_vs_trained.png")
plt.show()
print(f"Saved → {PLOTS_DIR}/baseline_vs_trained.png")

## 12. Save trained model

In [ ]:
# Save LoRA adapters locally
model.save_pretrained("./grpo-kaggle-agent-lora")
tokenizer.save_pretrained("./grpo-kaggle-agent-lora")
print("✅ LoRA adapters saved to ./grpo-kaggle-agent-lora")

# Optional: push to HF Hub
# model.push_to_hub("<your-username>/grpo-kaggle-agent")
# tokenizer.push_to_hub("<your-username>/grpo-kaggle-agent")

print("\n=== TRAINING SUMMARY ===")
print(f"Model          : {MODEL_NAME}")
print(f"Training steps : {train_result.global_step}")
print(f"Train loss     : {train_result.training_loss:.4f}")
print(f"Baseline score : {baseline_overall:.4f}")
print(f"Trained score  : {trained_overall:.4f}")
print(f"Improvement    : {delta_overall:+.4f}")
print(f"Plots saved to : ./{PLOTS_DIR}/")